In [1]:
#  Scenario: Package Delivery in a City
# Imagine you run a delivery company in a huge city with 100,000 houses. Each house has a
#  unique “address fingerprint” (like the 384‑dimensional vectors in your code).
# Now, when a customer places an order, you need to quickly find the nearest warehouse or
#  delivery hub to their house. Instead of checking every single house one by one (which would 
# be very slow), you use smart strategies:

# 🗺️ HNSW (Hierarchical Navigable Small World)
# - Think of it like a layered city map:
# - Top layer = highways (coarse overview).
# - Middle layer = main roads.
# - Bottom layer = all streets and houses.
# - To find the nearest warehouse, you start at the highway level, zoom into the right district, 
# then finally navigate down to the exact street.
# - This is what IndexHNSWFlat does: it builds a graph of connections so you can “hop” quickly 
# to the right neighborhood.

# 🏘️ IVF (Inverted File Index)
# - Imagine dividing the city into 256 neighborhoods (Voronoi cells).
# - First, figure out which neighborhood the customer’s house belongs to.
# - Then, only search inside that neighborhood instead of the whole city.
# - This is what IndexIVFFlat does: it clusters vectors and searches only in the most relevant 
# clusters.
# - nprobe = 10 means you don’t just check one neighborhood, but the 10 most likely ones —
#  balancing speed and accuracy.

# ⚡ In Plain Words
# - HNSW = zooming in layer by layer on a city map.
# - IVF = splitting the city into neighborhoods and searching locally.
# - Both methods help you find the closest match fast without scanning all 100,000 houses.

In [2]:
!pip install faiss-cpu  # Install faiss

import faiss
import numpy as np

d = 384       # embedding dimension
n = 100_000   # number of vectors

# ── HNSW index ──────────────────────────────
index = faiss.IndexHNSWFlat(d, 32)  # M=32 neighbours
index.hnsw.efConstruction = 200     # build quality
index.hnsw.efSearch = 100           # query quality

vectors = np.random.randn(n, d).astype("float32")
faiss.normalize_L2(vectors)         # L2 normalize
index.add(vectors)                  # add all vectors

query = np.random.randn(1, d).astype("float32")
faiss.normalize_L2(query)
D, I = index.search(query, k=10)    # top-10
print("Indices:", I[0])             # [2341, 8712, ...]
print("Distances:", D[0])           # cosine scores

# ── IVF index (faster for large scale) ──────
nlist = 256   # number of Voronoi cells
quantizer = faiss.IndexFlatL2(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist)
index_ivf.train(vectors)
index_ivf.add(vectors)
index_ivf.nprobe = 10               # cells to search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.0 MB/s eta 0:00:00:00:0100:01
Indices: [77672 50799 16209 31507 45377 23015 90806 38579 75846 49471]
Distances: [1.6052113 1.6194618 1.6289451 1.6339989 1.639224  1.6410453 1.6417756
 1.6434271 1.6501081 1.651151 ]
